# Cell 1: Import Required Libraries

In [2]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np


## Cell 2: Define Transformations and Load Dataset

In [3]:
# Define transformations and load dataset
transform = transforms.ToTensor()
dataset = datasets.MNIST(root='.', train=True, download=True, transform=transform)

# Split dataset into train and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:02<00:00, 4470699.35it/s]


Extracting .\MNIST\raw\train-images-idx3-ubyte.gz to .\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 102719.00it/s]


Extracting .\MNIST\raw\train-labels-idx1-ubyte.gz to .\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:18<00:00, 90627.65it/s] 


Extracting .\MNIST\raw\t10k-images-idx3-ubyte.gz to .\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 8986098.48it/s]

Extracting .\MNIST\raw\t10k-labels-idx1-ubyte.gz to .\MNIST\raw



## Cell 3: Define a Simple Neural Network

In [4]:
# Define a simple feedforward neural network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)  # Flatten image
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


## Cell 4: Instantiate Model, Loss Function, and Optimizer

In [5]:
# Instantiate model, loss, and optimizer
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## Cell 5: Define EarlyStopping Class

In [6]:
# Define EarlyStopping class
class EarlyStopping:
    def __init__(self, patience=3, delta=0.0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


## Cell 6: Training Loop with Early Stopping

In [7]:
# Training with early stopping
early_stopping = EarlyStopping(patience=5, delta=0.001)
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation phase
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss:.4f}")

    early_stopping(avg_val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break



Epoch 1, Validation Loss: 0.2105
Epoch 2, Validation Loss: 0.1448
Epoch 3, Validation Loss: 0.1217
Epoch 4, Validation Loss: 0.1049
Epoch 5, Validation Loss: 0.0997
Epoch 6, Validation Loss: 0.0927
Epoch 7, Validation Loss: 0.0891
Epoch 8, Validation Loss: 0.0861
Epoch 9, Validation Loss: 0.0838
Epoch 10, Validation Loss: 0.0802
Epoch 11, Validation Loss: 0.0857
Epoch 12, Validation Loss: 0.0953
Epoch 13, Validation Loss: 0.0840
Epoch 14, Validation Loss: 0.0857
Epoch 15, Validation Loss: 0.0891
Early stopping triggered.


## Cell 7: Add Accuracy Calculation

In [8]:
# Function to compute accuracy
def compute_accuracy(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total


## Cell 8: Training Loop with Accuracy and Model Saving

In [9]:
# Train with early stopping and save best model
early_stopping = EarlyStopping(patience=5, delta=0.001)
num_epochs = 50
best_model_path = "best_model.pth"

for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation phase
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = compute_accuracy(model, val_loader)

    print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")

    if avg_val_loss < early_stopping.best_loss - early_stopping.delta:
        torch.save(model.state_dict(), best_model_path)
        print("Model saved.")

    early_stopping(avg_val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break


Epoch 1 | Val Loss: 0.0974 | Val Accuracy: 0.9766
Model saved.
Epoch 2 | Val Loss: 0.0990 | Val Accuracy: 0.9768
Epoch 3 | Val Loss: 0.0947 | Val Accuracy: 0.9791
Model saved.
Epoch 4 | Val Loss: 0.1052 | Val Accuracy: 0.9780
Epoch 5 | Val Loss: 0.1242 | Val Accuracy: 0.9742
Epoch 6 | Val Loss: 0.1024 | Val Accuracy: 0.9790
Epoch 7 | Val Loss: 0.1031 | Val Accuracy: 0.9790
Epoch 8 | Val Loss: 0.1126 | Val Accuracy: 0.9768
Early stopping triggered.


## Cell 9: Load the Best Saved Model

In [10]:
# Load the best saved model
best_model = SimpleNN()
best_model.load_state_dict(torch.load(best_model_path))
best_model.eval()

# Evaluate final accuracy on validation set
final_accuracy = compute_accuracy(best_model, val_loader)
print(f"Final validation accuracy from best saved model: {final_accuracy:.4f}")


Final validation accuracy from best saved model: 0.9791
